# CWD-PHFT — Forward Pass Test Notebook

Validates the full model pipeline by importing from the `python/model` package.
Same test structure as `python/tests/test_model.py`, but with printed diagnostics
instead of asserts. Run all cells top to bottom; every section prints a PASS/FAIL
style diagnostic.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath(''), '..', 'python'))

import torch
from model.cwd_phft import CWDPHFT
from model.lorentz import LorentzManifold

torch.manual_seed(0)
print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()}")


## 1. Build a small model

Smaller config than the established hyperparameters so the notebook runs fast on CPU.


In [ ]:
model = CWDPHFT(
    vocab_size=1000, dim=64, num_layers=2, num_heads=4,
    mem_size=16, field_decay=0.997, mem_momentum=0.99,
    fractal_iters=2, num_modes=4,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built: {n_params:,} parameters")


## 2. Forward pass shape + NaN check

In [ ]:
ids = torch.randint(0, 1000, (2, 64))
logits = model(ids)
print(f"logits shape: {tuple(logits.shape)}  (expect (2, 64, 1000))")
print(f"NaN in logits: {torch.isnan(logits).any().item()}  (expect False)")
print(f"logit range: [{logits.min().item():.3f}, {logits.max().item():.3f}]")


## 3. Energy head

In [ ]:
logits, energy = model(ids, return_energy=True)
print(f"energy shape: {tuple(energy.shape)}  (expect (2, 64))")
print(f"mean energy: {energy.mean().item():.4f}")
print(f"sequence energy per batch: {energy.mean(dim=-1).tolist()}")


## 4. Epsilon manifold — modes produce different outputs

In [ ]:
ids_small = torch.randint(0, 1000, (1, 32))
outs = []
for m in range(4):
    model.detach_persistent_state()
    outs.append(model(ids_small, mode=m))
for i in range(4):
    for j in range(i + 1, 4):
        same = torch.allclose(outs[i], outs[j], atol=1e-4)
        print(f"mode {i} vs mode {j}: identical={same}  (expect False)")


## 5. Persistent field — state evolves across batches, reset clears it

In [ ]:
model.reset_persistent_state()
states = []
for step in range(3):
    _ = model(torch.randint(0, 1000, (1, 32)))
    states.append(model.field.identity_state.clone().detach())
    model.detach_persistent_state()
    print(f"step {step}: field metrics = {model.field.last_metrics}")

for i in range(len(states) - 1):
    moved = not torch.allclose(states[i], states[i + 1], atol=1e-7)
    print(f"identity state moved between step {i} and {i+1}: {moved}  (expect True)")

before = model.field.identity_state.clone()
model.reset_persistent_state()
after = model.field.identity_state.clone()
print(f"reset changed identity state: {not torch.allclose(before, after, atol=1e-5)}  (expect True)")
print(f"step_count after reset: {model.field.step_count.item()}  (expect 0)")


## 6. Gradient flow through the full stack

In [ ]:
model.train()
model.reset_persistent_state()
ids = torch.randint(0, 1000, (2, 32))
logits, energy = model(ids, return_energy=True)
loss = torch.nn.functional.cross_entropy(
    logits[:, :-1].reshape(-1, 1000), ids[:, 1:].reshape(-1)
) + 0.01 * energy.mean()
loss.backward()

grad_sum = sum(
    p.grad.abs().sum().item()
    for p in model.layers[0].attn.parameters() if p.grad is not None
)
print(f"loss: {loss.item():.4f}  (expect ~ln(1000) ~ 6.9 at init)")
print(f"attention grad abs-sum: {grad_sum:.6f}  (expect > 0)")
model.zero_grad()
model.detach_persistent_state()


## 7. Checkpoint round-trip

In [ ]:
import tempfile

model.eval()
with tempfile.TemporaryDirectory() as td:
    ckpt_path = os.path.join(td, 'test.pt')
    torch.save({
        'model_state_dict': model.state_dict(),
        'field_state': model.get_field_state_dict(),
    }, ckpt_path)

    model2 = CWDPHFT(
        vocab_size=1000, dim=64, num_layers=2, num_heads=4,
        mem_size=16, field_decay=0.997, mem_momentum=0.99,
        fractal_iters=2, num_modes=4,
    )
    ckpt = torch.load(ckpt_path, weights_only=False)
    model2.load_state_dict(ckpt['model_state_dict'])
    model2.load_field_state_dict(ckpt['field_state'])
    model2.eval()

    ids = torch.randint(0, 1000, (1, 16))
    with torch.no_grad():
        out1 = model(ids)
        out2 = model2(ids)
    max_diff = (out1 - out2).abs().max().item()
print(f"max logit diff after checkpoint round-trip: {max_diff:.2e}  (expect < 1e-5)")


## 8. Component diagnostics — Lorentz, octonion, Fano mask, Hamming

In [ ]:
from model.octonion import OctonionLinear
from model.attention import FanoSparseAttention
from model.hamming import HammingEmbedding

# Lorentz round-trip (in-domain)
v = torch.randn(4, 512)
v = v / v.norm(dim=-1, keepdim=True) * 1.0
err = (LorentzManifold.logmap0(LorentzManifold.expmap0(v)) - v).norm(dim=-1).mean().item()
print(f"Lorentz round-trip error (norm=1.0): {err:.2e}  (target < 1e-3)")

# Octonion compression
std_params = 512 * 512 + 512
oct_params = sum(p.numel() for p in OctonionLinear(512, 512).parameters())
print(f"Octonion compression: {std_params/oct_params:.2f}x  (target ~8x)")

# Fano mask density
attn = FanoSparseAttention(512, num_heads=8)
for T in [64, 256]:
    print(f"Fano mask density T={T}: {attn.density(T):.4f}  (target ~0.4286)")

# Hamming syndrome
hemb = HammingEmbedding(1000, 64)
ids = torch.randint(0, 1000, (2, 16))
print(f"Hamming syndrome magnitude (clean tokens): {hemb.syndrome_magnitude(ids):.4f}")
